# Phase 7A — Submission Assembly and Validation

This notebook freezes the validated Phase 3–6 evidence used by the final submission. It does not recalculate player scores or alter the official rankings. It creates the canonical player bundle, claim and terminology registries, design tokens, submission structure, manifests, and Phase 7A validation gates.

In [ ]:
from pathlib import Path
import hashlib
import json

import numpy as np
import pandas as pd

ROOT = Path.cwd()
OUT = ROOT / 'phase7_outputs'
OUT.mkdir(exist_ok=True)
PHASE7_MODEL_VERSION = 'submission_foundation_v1'
ROLE_ORDER = {'Batter': 0, 'Bowler': 1}

def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def csv_record(path: Path, purpose: str) -> dict:
    frame = pd.read_csv(path)
    return {
        'source_file': path.relative_to(ROOT).as_posix(),
        'sha256': sha256_file(path),
        'rows': len(frame),
        'columns': len(frame.columns),
        'purpose': purpose,
        'frozen_for_phase7': True,
        'phase7_model_version': PHASE7_MODEL_VERSION,
    }


In [ ]:
SOURCE_SPECS = {
    'phase3_outputs/phase3_output_manifest.csv': 'Phase 3 artifact hashes',
    'phase3_outputs/phase3_model_specification.csv': 'Locked structural-pressure definition',
    'phase3_outputs/phase3_validation_gates.csv': 'Phase 3 validation status',
    'phase4_outputs/phase4_output_manifest.csv': 'Phase 4 artifact hashes and Phase 3 source hash',
    'phase4_outputs/phase4_methodology_specification.csv': 'Locked player-scoring rules',
    'phase4_outputs/phase4_batter_format_metrics.csv': 'Batter format ratings and eligibility',
    'phase4_outputs/phase4_bowler_format_metrics.csv': 'Bowler format ratings and eligibility',
    'phase4_outputs/phase4_batter_combined_scores.csv': 'Batter combined ratings',
    'phase4_outputs/phase4_bowler_combined_scores.csv': 'Bowler combined ratings',
    'phase4_outputs/phase4_candidate_summary.csv': 'Reliable candidate counts',
    'phase4_outputs/phase4_validation_gates.csv': 'Phase 4 validation status',
    'phase5_outputs/phase5_output_manifest.csv': 'Phase 5 artifact hashes and source hash',
    'phase5_outputs/phase5_methodology_specification.csv': 'Locked ranking and uncertainty rules',
    'phase5_outputs/phase5_official_rankings.csv': 'Official ranking order',
    'phase5_outputs/phase5_bootstrap_summary.csv': 'Bootstrap uncertainty evidence',
    'phase5_outputs/phase5_robustness_summary.csv': 'Deterministic robustness evidence',
    'phase5_outputs/phase5_scenario_rankings.csv': 'Scenario ranking evidence',
    'phase5_outputs/phase5_self_influence.csv': 'Leave-one-player-out benchmark evidence',
    'phase5_outputs/phase5_boundary_and_provisional_candidates.csv': 'Boundary and provisional candidates',
    'phase5_outputs/phase5_validation_gates.csv': 'Phase 5 validation status',
    'phase6_outputs/phase6_output_manifest.csv': 'Phase 6 artifact hashes and source hash',
    'phase6_outputs/phase6_selected_player_evidence.csv': 'Canonical selected-player explanations',
    'phase6_outputs/phase6_selected_format_evidence.csv': 'Selected-player format evidence',
    'phase6_outputs/phase6_pressure_vs_ordinary_evidence.csv': 'Pressure versus ordinary evidence',
    'phase6_outputs/phase6_uncertainty_and_robustness.csv': 'Selected-player uncertainty summary',
    'phase6_outputs/phase6_challenger_and_provisional_appendix.csv': 'Challenger and provisional appendix',
    'phase6_outputs/phase6_validation_gates.csv': 'Phase 6 validation status',
}

missing_sources = [name for name in SOURCE_SPECS if not (ROOT / name).exists()]
if missing_sources:
    raise FileNotFoundError(f'Missing required Phase 7 source files: {missing_sources}')

source_manifest = pd.DataFrame([
    csv_record(ROOT / name, purpose) for name, purpose in SOURCE_SPECS.items()
]).sort_values('source_file').reset_index(drop=True)
source_manifest.to_csv(OUT / 'phase7_source_manifest.csv', index=False)
SOURCE_HASHES = dict(zip(source_manifest['source_file'], source_manifest['sha256']))
source_manifest[['source_file', 'rows', 'columns', 'sha256']].head()


In [ ]:
phase3_spec = pd.read_csv(ROOT / 'phase3_outputs/phase3_model_specification.csv')
phase4_spec = pd.read_csv(ROOT / 'phase4_outputs/phase4_methodology_specification.csv')
phase5_spec = pd.read_csv(ROOT / 'phase5_outputs/phase5_methodology_specification.csv')
phase5_rankings = pd.read_csv(ROOT / 'phase5_outputs/phase5_official_rankings.csv')
phase6_players = pd.read_csv(ROOT / 'phase6_outputs/phase6_selected_player_evidence.csv')
phase6_formats = pd.read_csv(ROOT / 'phase6_outputs/phase6_selected_format_evidence.csv')
phase6_pressure = pd.read_csv(ROOT / 'phase6_outputs/phase6_pressure_vs_ordinary_evidence.csv')
phase6_uncertainty = pd.read_csv(ROOT / 'phase6_outputs/phase6_uncertainty_and_robustness.csv')

official = phase5_rankings.loc[phase5_rankings['official_top10'].eq(True)].copy()
official['_role_order'] = official['role'].map(ROLE_ORDER)
official = official.sort_values(['_role_order', 'rank', 'player_id']).drop(columns='_role_order').reset_index(drop=True)
selected = phase6_players.copy()
selected['_role_order'] = selected['role'].map(ROLE_ORDER)
selected = selected.sort_values(['_role_order', 'rank', 'player_id']).drop(columns='_role_order').reset_index(drop=True)

comparison_columns = ['role', 'rank', 'player_id', 'player_name', 'qualifying_formats', 'combined_rating']
official_compare = official[comparison_columns].reset_index(drop=True)
selected_compare = selected[comparison_columns].reset_index(drop=True)
text_columns = ['role', 'player_name', 'qualifying_formats']
integer_columns = ['rank', 'player_id']
identity_match = all(official_compare[col].equals(selected_compare[col]) for col in text_columns + integer_columns)
rating_match = np.allclose(official_compare['combined_rating'], selected_compare['combined_rating'], atol=1e-10, rtol=0)

ranking_signature_frame = official_compare.copy()
ranking_signature_frame['combined_rating'] = ranking_signature_frame['combined_rating'].map(lambda value: f'{value:.10f}')
ranking_signature = hashlib.sha256(ranking_signature_frame.to_csv(index=False).encode('utf-8')).hexdigest()
ranking_signature


In [ ]:
canonical = selected.copy()
canonical.insert(0, 'submission_locked', True)
canonical.insert(1, 'official_ranking_signature', ranking_signature)
canonical['phase7_model_version'] = PHASE7_MODEL_VERSION
canonical.to_csv(OUT / 'phase7_canonical_player_evidence.csv', index=False)

bundle = {
    'metadata': {
        'phase7_model_version': PHASE7_MODEL_VERSION,
        'official_period': '2022-2026',
        'recent_form_weight': 0,
        'format_rule': 'IPL and T20I evaluated separately; 50/50 when both are Reliable; one Reliable format used alone; missing is not zero',
        'pressure_model': 'structural_pressure_v1',
        'ranking_rule': 'unrounded combined rating descending; player_id ascending tie-break',
        'official_ranking_signature': ranking_signature,
        'source_hashes': SOURCE_HASHES,
    },
    'players': canonical.where(pd.notna(canonical), None).to_dict(orient='records'),
}
(OUT / 'phase7_canonical_bundle.json').write_text(json.dumps(bundle, indent=2, ensure_ascii=False), encoding='utf-8')
canonical[['role', 'rank', 'player_name', 'qualifying_formats', 'combined_rating']]


In [ ]:
def rule(frame: pd.DataFrame, item: str) -> str:
    match = frame.loc[frame['item'].eq(item), 'locked_rule']
    if len(match) != 1:
        raise ValueError(f'Expected exactly one locked rule for {item}; found {len(match)}')
    return str(match.iloc[0])

claims = [
    {'claim_id': 'METHOD_PERIOD', 'category': 'Methodology', 'subject_id': 'GLOBAL', 'claim_text': 'All eligible evidence from 2022 through 2026 is used.', 'value_text': rule(phase4_spec, 'official_period'), 'source_file': 'phase4_outputs/phase4_methodology_specification.csv', 'source_field': 'official_period.locked_rule'},
    {'claim_id': 'METHOD_RECENCY', 'category': 'Methodology', 'subject_id': 'GLOBAL', 'claim_text': 'No recent-form weight is applied.', 'value_text': '0%', 'source_file': 'phase4_outputs/phase4_methodology_specification.csv', 'source_field': 'official_period.note'},
    {'claim_id': 'METHOD_FORMAT', 'category': 'Methodology', 'subject_id': 'GLOBAL', 'claim_text': 'IPL and T20I are evaluated separately before the validated format-combination rule is applied.', 'value_text': rule(phase4_spec, 'format_combination'), 'source_file': 'phase4_outputs/phase4_methodology_specification.csv', 'source_field': 'format_combination.locked_rule'},
    {'claim_id': 'PRESSURE_CONSTRUCT', 'category': 'Pressure', 'subject_id': 'GLOBAL', 'claim_text': 'Pressure is a pre-delivery structural state.', 'value_text': rule(phase3_spec, 'construct'), 'source_file': 'phase3_outputs/phase3_model_specification.csv', 'source_field': 'construct.locked_rule'},
    {'claim_id': 'PRESSURE_INDEX', 'category': 'Pressure', 'subject_id': 'GLOBAL', 'claim_text': 'The structural-pressure index is the equal mean of wicket strain, recent wicket shock, and stage urgency.', 'value_text': rule(phase3_spec, 'structural_pressure_index'), 'source_file': 'phase3_outputs/phase3_model_specification.csv', 'source_field': 'structural_pressure_index.locked_rule'},
    {'claim_id': 'PRESSURE_BANDS', 'category': 'Pressure', 'subject_id': 'GLOBAL', 'claim_text': 'High pressure is the top 20 percent within format and innings.', 'value_text': rule(phase3_spec, 'pressure_category'), 'source_file': 'phase3_outputs/phase3_model_specification.csv', 'source_field': 'pressure_category.locked_rule'},
    {'claim_id': 'RANKING_RULE', 'category': 'Ranking', 'subject_id': 'GLOBAL', 'claim_text': 'Official order uses unrounded combined ratings with player ID as the deterministic tie-break.', 'value_text': rule(phase5_spec, 'ranking_rule'), 'source_file': 'phase5_outputs/phase5_methodology_specification.csv', 'source_field': 'ranking_rule.locked_rule'},
    {'claim_id': 'BOOTSTRAP_FIXED', 'category': 'Uncertainty', 'subject_id': 'GLOBAL', 'claim_text': 'Fixed-benchmark uncertainty uses 5,000 stratified match replicates.', 'value_text': rule(phase5_spec, 'fixed_bootstrap'), 'source_file': 'phase5_outputs/phase5_methodology_specification.csv', 'source_field': 'fixed_bootstrap.locked_rule'},
    {'claim_id': 'BOOTSTRAP_NESTED', 'category': 'Uncertainty', 'subject_id': 'GLOBAL', 'claim_text': 'Nested-benchmark uncertainty uses 1,000 stratified match replicates.', 'value_text': rule(phase5_spec, 'nested_bootstrap'), 'source_file': 'phase5_outputs/phase5_methodology_specification.csv', 'source_field': 'nested_bootstrap.locked_rule'},
    {'claim_id': 'BOUNDARY_RULE', 'category': 'Uncertainty', 'subject_id': 'GLOBAL', 'claim_text': 'Official selections below the 75 percent threshold in either bootstrap are boundary cases, not automatic replacements.', 'value_text': rule(phase5_spec, 'boundary_top10'), 'source_file': 'phase5_outputs/phase5_methodology_specification.csv', 'source_field': 'boundary_top10.locked_rule'},
]

for row in canonical.itertuples(index=False):
    subject = f'{row.role.upper()}_{int(row.player_id)}'
    claims.extend([
        {'claim_id': f'{subject}_SELECTION', 'category': 'Player selection', 'subject_id': subject, 'claim_text': row.selection_reason, 'value_text': f'Rank {int(row.rank)}; rating {row.combined_rating:.8f}', 'source_file': 'phase6_outputs/phase6_selected_player_evidence.csv', 'source_field': 'selection_reason'},
        {'claim_id': f'{subject}_FORMAT', 'category': 'Format interpretation', 'subject_id': subject, 'claim_text': row.format_interpretation, 'value_text': row.qualifying_formats, 'source_file': 'phase6_outputs/phase6_selected_player_evidence.csv', 'source_field': 'format_interpretation'},
        {'claim_id': f'{subject}_PRESSURE', 'category': 'Pressure interpretation', 'subject_id': subject, 'claim_text': row.pressure_interpretation, 'value_text': 'Approved narrative', 'source_file': 'phase6_outputs/phase6_selected_player_evidence.csv', 'source_field': 'pressure_interpretation'},
        {'claim_id': f'{subject}_UNCERTAINTY', 'category': 'Uncertainty interpretation', 'subject_id': subject, 'claim_text': row.uncertainty_interpretation, 'value_text': row.robustness_class, 'source_file': 'phase6_outputs/phase6_selected_player_evidence.csv', 'source_field': 'uncertainty_interpretation'},
    ])

claim_registry = pd.DataFrame(claims)
claim_registry['locked'] = True
claim_registry['phase7_model_version'] = PHASE7_MODEL_VERSION
claim_registry.to_csv(OUT / 'phase7_claim_registry.csv', index=False)
claim_registry.groupby('category').size().rename('claims')


In [ ]:
terminology_rows = [
    ('Structural pressure', 'A pre-delivery state built from wicket strain, recent wicket shock, and innings-stage urgency.', 'Do not call it win probability, required-run-rate pressure, or a complete scoreboard-pressure model.', 'phase3_outputs/phase3_model_specification.csv'),
    ('High pressure', 'Records above the 80th percentile of the structural-pressure index within format and innings.', 'Do not classify pressure from the current ball outcome.', 'phase3_outputs/phase3_model_specification.csv'),
    ('Ordinary', 'Low- and Medium-pressure records combined for Phase 6 evidence comparisons.', 'Do not imply that every ordinary delivery is easy or low consequence.', 'phase6_outputs/phase6_pressure_vs_ordinary_evidence.csv'),
    ('Combined rating', 'The validated player rating after separate IPL and T20I normalization and the locked format-combination rule.', 'Do not assign zero to a missing or unreliable format.', 'phase4_outputs/phase4_methodology_specification.csv'),
    ('Reliable', 'A format sample that passes all locked Phase 4 eligibility gates and may contribute to the official score.', 'Do not use Provisional or Insufficient evidence in an official score.', 'phase4_outputs/phase4_batter_format_metrics.csv'),
    ('Provisional', 'Evidence that is informative but fails at least one Reliable sample gate.', 'Do not present it as equivalent to Reliable evidence.', 'phase6_outputs/phase6_selected_format_evidence.csv'),
    ('Insufficient', 'Evidence that fails the minimum sample requirements for official use.', 'Do not use it to fill a leaderboard.', 'phase6_outputs/phase6_selected_format_evidence.csv'),
    ('Pressure gradient', 'The continuous association between the pressure percentile and context-adjusted performance.', 'Do not treat it as the same quantity as the High-pressure quality metric.', 'phase4_outputs/phase4_methodology_specification.csv'),
    ('Run suppression', 'Context-adjusted batter-run suppression on no-extras delivery records.', 'Do not call it conventional economy rate.', 'phase4_outputs/phase4_methodology_specification.csv'),
    ('Top-10 bootstrap frequency', 'The share of match-resampling replicates in which a player enters the recalculated Top 10.', 'Do not describe it as the probability that the player is objectively Top 10.', 'phase5_outputs/phase5_methodology_specification.csv'),
    ('Boundary Top 10', 'An official full-data Top-10 player below the 75 percent inclusion threshold in at least one bootstrap mode.', 'Do not automatically replace the player or call the official rank invalid.', 'phase5_outputs/phase5_methodology_specification.csv'),
    ('Qualifying formats', 'The Reliable format evidence used in the official combined rating.', 'Do not interpret an omitted format as a zero score.', 'phase6_outputs/phase6_selected_player_evidence.csv'),
]
terminology = pd.DataFrame(terminology_rows, columns=['term', 'approved_definition', 'prohibited_or_caution', 'source_file'])
terminology['phase7_model_version'] = PHASE7_MODEL_VERSION
terminology.to_csv(OUT / 'phase7_terminology_registry.csv', index=False)

design_token_rows = [
    ('Color', 'midnight', '#07111F', 'Primary dark background and strongest text'),
    ('Color', 'slate', '#132238', 'Secondary dark surface and section bands'),
    ('Color', 'teal', '#20C7B6', 'Validated evidence and interactive focus'),
    ('Color', 'gold', '#F5B942', 'Ranks, winners, and primary highlights'),
    ('Color', 'coral', '#FF6B5E', 'Warnings, uncertainty, and limitations'),
    ('Color', 'off_white', '#F4F7FB', 'Light report surface and dark-background text'),
    ('Color', 'ink', '#172033', 'Primary body text on light surfaces'),
    ('Color', 'muted', '#6B7280', 'Secondary labels and explanatory text'),
    ('Typography', 'display_font', 'Avenir Next', 'Titles and display numerals; Arial fallback'),
    ('Typography', 'body_font', 'Avenir Next', 'Body text and tables; Arial fallback'),
    ('Typography', 'minimum_presentation_body', '17 pt', 'Minimum presentation body text'),
    ('Typography', 'report_body', '9.5-10.5 pt', 'A4 report body range'),
    ('Layout', 'presentation_ratio', '16:9', 'Presentation and dashboard-demo frame'),
    ('Layout', 'report_page', 'A4 portrait', 'Primary report page size'),
    ('Chart', 'validated_series', '#20C7B6', 'Reliable or validated evidence'),
    ('Chart', 'ranking_series', '#F5B942', 'Official rank or selected player'),
    ('Chart', 'uncertainty_series', '#FF6B5E', 'Uncertainty and limitations'),
    ('Accessibility', 'encoding', 'Color plus text or shape', 'Never use color as the only signal'),
]
design_tokens = pd.DataFrame(design_token_rows, columns=['token_group', 'token_name', 'value', 'usage'])
design_tokens['phase7_model_version'] = PHASE7_MODEL_VERSION
design_tokens.to_csv(OUT / 'phase7_design_tokens.csv', index=False)

structure_rows = [
    ('7A', 'Notebook', 'Phase_7_Submission_Assembly_and_Validation.ipynb', True, 'Complete', 'Phase 7 source freeze and validation'),
    ('7A', 'Report', 'Phase 7A Submission Foundation Report.md', True, 'Complete', 'Foundation decisions and handoff'),
    ('7A', 'Specification', 'Phase 7 Visual Identity Specification.md', True, 'Complete', 'Shared artifact design rules'),
    ('7A', 'Data bundle', 'phase7_outputs/', True, 'Complete', 'Canonical claims, evidence, manifests, and gates'),
    ('7B', 'Dashboard data', 'phase7_dashboard_data/', True, 'Planned', 'Format and national-team filtering data'),
    ('7C', 'Dashboard', 'dashboard/', True, 'Planned', 'Published interactive leaderboard'),
    ('7D', 'Report', 'final_submission/SupeRR_Selector_Methodology_Report.pdf', True, 'Planned', 'Final methodology report'),
    ('7E', 'Presentation', 'final_submission/SupeRR_Selector_Presentation.pptx', True, 'Planned', 'Editable judging presentation'),
    ('7F', 'QA package', 'final_submission/manifest/', True, 'Planned', 'Cross-artifact checks and submission sheet'),
    ('7G', 'Video', 'final_submission/SupeRR_Selector_Journey.mp4', True, 'Deferred', 'Separate video plan required before implementation'),
]
submission_structure = pd.DataFrame(structure_rows, columns=['subphase', 'artifact_type', 'planned_path', 'required_for_project', 'status', 'purpose'])
submission_structure['phase7_model_version'] = PHASE7_MODEL_VERSION
submission_structure.to_csv(OUT / 'phase7_submission_structure.csv', index=False)


In [ ]:
validation_rows = []

def add_gate(check_name: str, passed: bool, observed, acceptance_rule: str) -> None:
    validation_rows.append({
        'check_name': check_name,
        'passed': bool(passed),
        'observed': observed,
        'acceptance_rule': acceptance_rule,
        'status': 'PASS' if passed else 'FAIL',
        'phase7_model_version': PHASE7_MODEL_VERSION,
    })

upstream_status = {}
for phase in (3, 4, 5, 6):
    gates = pd.read_csv(ROOT / f'phase{phase}_outputs/phase{phase}_validation_gates.csv')
    upstream_status[phase] = bool(gates['status'].eq('PASS').all())

manifest_missing = []
manifest_hash_mismatches = []
manifest_entries_checked = 0
for phase in (3, 4, 5, 6):
    declared = pd.read_csv(ROOT / f'phase{phase}_outputs/phase{phase}_output_manifest.csv')
    for row in declared.itertuples(index=False):
        candidates = [ROOT / f'phase{phase}_outputs' / row.artifact, ROOT / row.artifact]
        artifact_path = next((path for path in candidates if path.exists()), None)
        if artifact_path is None:
            manifest_missing.append(f'phase{phase}:{row.artifact}')
            continue
        manifest_entries_checked += 1
        current_hash = sha256_file(artifact_path)
        if current_hash != row.sha256:
            manifest_hash_mismatches.append(f'phase{phase}:{row.artifact}')

role_counts = official.groupby('role').size().to_dict()
rank_sequences = official.groupby('role')['rank'].apply(list).to_dict()
format_counts = phase6_formats.groupby(['role', 'player_id']).size()
used_format_rows = phase6_formats.loc[phase6_formats['used_in_official_score'].eq(True)]
excluded_format_rows = phase6_formats.loc[phase6_formats['used_in_official_score'].eq(False)]

add_gate('required_sources_present', not missing_sources, len(SOURCE_SPECS) - len(missing_sources), f'{len(SOURCE_SPECS)} required sources present')
add_gate('upstream_validation_pass', all(upstream_status.values()), upstream_status, 'Phases 3, 4, 5, and 6 validation gates all PASS')
add_gate('upstream_output_manifests_match', not manifest_missing and not manifest_hash_mismatches, {'entries_checked': manifest_entries_checked, 'missing': manifest_missing, 'hash_mismatches': manifest_hash_mismatches}, 'Every artifact declared by the Phase 3-6 manifests exists and matches its stored SHA-256')
add_gate('official_selection_count', role_counts == {'Batter': 10, 'Bowler': 10}, role_counts, 'Exactly 10 official players per role')
add_gate('official_ids_unique_within_role', not official.duplicated(['role', 'player_id']).any(), int(official.duplicated(['role', 'player_id']).sum()), 'Zero duplicate player IDs within a role')
add_gate('official_rank_sequence', all(values == list(range(1, 11)) for values in rank_sequences.values()), rank_sequences, 'Ranks 1 through 10 exactly once per role')
add_gate('phase5_phase6_identity_match', identity_match, identity_match, 'Role, rank, player ID, name, and qualifying formats match exactly')
add_gate('phase5_phase6_rating_match', rating_match, float(np.max(np.abs(official_compare['combined_rating'] - selected_compare['combined_rating']))), 'Maximum absolute rating difference <= 1e-10')
add_gate('no_recent_form_weight', rule(phase5_spec, 'official_period') == 'FULL_2022_2026_NO_RECENCY_WEIGHT', rule(phase5_spec, 'official_period'), 'Locked period explicitly has no recency weight')
add_gate('all_official_boundary_class_preserved', canonical['robustness_class'].eq('BOUNDARY_TOP10').all(), canonical['robustness_class'].value_counts().to_dict(), 'Every official selection retains its Phase 6 robustness class')
add_gate('two_format_rows_per_player', format_counts.eq(2).all() and len(format_counts) == 20, {'players': len(format_counts), 'minimum_rows': int(format_counts.min()), 'maximum_rows': int(format_counts.max())}, 'Exactly two format-evidence rows for every selected player')
add_gate('used_formats_reliable', used_format_rows['evidence_status'].eq('Reliable').all(), int((~used_format_rows['evidence_status'].eq('Reliable')).sum()), 'All formats used in official scores are Reliable')
add_gate('excluded_formats_explained', excluded_format_rows['format_use_explanation'].fillna('').str.len().gt(0).all(), int(excluded_format_rows['format_use_explanation'].fillna('').str.len().eq(0).sum()), 'Every excluded format has a non-empty explanation')
add_gate('canonical_essential_fields_complete', canonical[['role', 'rank', 'player_id', 'player_name', 'combined_rating', 'selection_reason', 'format_interpretation', 'pressure_interpretation', 'uncertainty_interpretation']].notna().all().all(), int(canonical[['role', 'rank', 'player_id', 'player_name', 'combined_rating']].isna().sum().sum()), 'No missing essential canonical fields')
add_gate('claim_ids_unique', not claim_registry['claim_id'].duplicated().any(), int(claim_registry['claim_id'].duplicated().sum()), 'Zero duplicate claim IDs')
add_gate('player_claim_coverage', len(claim_registry.loc[claim_registry['subject_id'].ne('GLOBAL')]) == 80, len(claim_registry.loc[claim_registry['subject_id'].ne('GLOBAL')]), 'Four approved player claims for each of 20 selections')
add_gate('terminology_registry_complete', len(terminology) >= 12 and terminology[['term', 'approved_definition', 'prohibited_or_caution', 'source_file']].notna().all().all(), len(terminology), 'At least 12 complete controlled terms')
add_gate('design_tokens_complete', {'midnight', 'teal', 'gold', 'coral', 'off_white'}.issubset(set(design_tokens['token_name'])), sorted(design_tokens['token_name'].tolist()), 'Required shared colors and accessibility token present')
add_gate('submission_structure_covers_all_subphases', set(submission_structure['subphase']) == {'7A', '7B', '7C', '7D', '7E', '7F', '7G'}, sorted(submission_structure['subphase'].unique().tolist()), 'Subphases 7A through 7G represented')
add_gate('bundle_player_count', len(bundle['players']) == 20, len(bundle['players']), 'Canonical JSON bundle contains 20 selected players')

validation = pd.DataFrame(validation_rows)
validation.to_csv(OUT / 'phase7_validation_gates.csv', index=False)

completion_summary = pd.DataFrame([
    {'metric': 'official_batters', 'value': role_counts.get('Batter', 0)},
    {'metric': 'official_bowlers', 'value': role_counts.get('Bowler', 0)},
    {'metric': 'source_files_frozen', 'value': len(source_manifest)},
    {'metric': 'approved_claims', 'value': len(claim_registry)},
    {'metric': 'controlled_terms', 'value': len(terminology)},
    {'metric': 'design_tokens', 'value': len(design_tokens)},
    {'metric': 'validation_gates', 'value': len(validation)},
    {'metric': 'validation_failures', 'value': int(validation['status'].eq('FAIL').sum())},
    {'metric': 'official_ranking_signature', 'value': ranking_signature},
])
completion_summary['phase7_model_version'] = PHASE7_MODEL_VERSION
completion_summary.to_csv(OUT / 'phase7_completion_summary.csv', index=False)

validation[['check_name', 'status', 'observed']]


In [ ]:
generated_specs = {
    'phase7_canonical_player_evidence.csv': ('Canonical selected-player evidence', len(canonical)),
    'phase7_canonical_bundle.json': ('Machine-readable canonical Phase 7 bundle', len(bundle['players'])),
    'phase7_claim_registry.csv': ('Approved claims and provenance', len(claim_registry)),
    'phase7_terminology_registry.csv': ('Controlled terminology and cautions', len(terminology)),
    'phase7_design_tokens.csv': ('Shared visual design tokens', len(design_tokens)),
    'phase7_submission_structure.csv': ('Subphase artifact structure', len(submission_structure)),
    'phase7_source_manifest.csv': ('Frozen Phase 3-6 source manifest', len(source_manifest)),
    'phase7_validation_gates.csv': ('Phase 7A validation results', len(validation)),
    'phase7_completion_summary.csv': ('Phase 7A completion metrics', len(completion_summary)),
}
document_specs = {
    'Phase 7A Submission Foundation Report.md': 'Phase 7A implementation report',
    'Phase 7 Visual Identity Specification.md': 'Shared visual identity specification',
}
output_rows = []
for filename, (purpose, rows) in generated_specs.items():
    path = OUT / filename
    output_rows.append({'output_file': path.relative_to(ROOT).as_posix(), 'sha256': sha256_file(path), 'rows_or_items': rows, 'purpose': purpose, 'phase7_model_version': PHASE7_MODEL_VERSION})
for filename, purpose in document_specs.items():
    path = ROOT / filename
    if path.exists():
        output_rows.append({'output_file': path.name, 'sha256': sha256_file(path), 'rows_or_items': 1, 'purpose': purpose, 'phase7_model_version': PHASE7_MODEL_VERSION})
output_manifest = pd.DataFrame(output_rows).sort_values('output_file').reset_index(drop=True)
output_manifest.to_csv(OUT / 'phase7_output_manifest.csv', index=False)

failed = validation.loc[validation['status'].eq('FAIL'), 'check_name'].tolist()
if failed:
    raise AssertionError(f'Phase 7A validation failed: {failed}')

print(f'Phase 7A complete: {len(validation)} validation gates passed.')
print(f'Official ranking signature: {ranking_signature}')
output_manifest
